# Silver normalization

Reads only immutable Bronze files already landed in the attached lakehouse, applies the reusable Silver contracts, and upserts schema-qualified Delta tables.

In [ ]:
# ruff: noqa: E501, F821
run_id = "manual"
run_mode = "daily"
scope_kind = "organization"
scope_slug = ""
entity_type = "copilot_usage"
start_date = ""
end_date = ""
lakehouse_files_root = "/lakehouse/default/Files"
bronze_folder = "bronze"
silver_schema = "silver"
audit_schema = "audit"


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql.types import StringType, StructField, StructType

from copilot_metrics_fabric.silver import (
    NormalizedBatch,
    SourceContext,
    normalize_ndjson,
    select_latest_manifests,
)
from copilot_metrics_fabric.silver_spark import create_dataframes, merge_dataframes

AUDIT_COLUMNS = ("run_id", "stage", "status", "started_at", "completed_at", "scope_kind", "scope_slug", "start_date", "end_date", "validation_status", "details_json")

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def write_audit(status: str, started_at: str, validation_status: str, details: dict) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{audit_schema}`")
    schema = StructType([StructField(name, StringType(), False) for name in AUDIT_COLUMNS])
    row = (run_id, "silver", status, started_at, utc_now(), scope_kind, scope_slug, start_date, end_date, validation_status, json.dumps(details, sort_keys=True))
    spark.createDataFrame([row], schema).write.format("delta").mode("append").saveAsTable(f"{audit_schema}.pipeline_run_results")


In [ ]:
started_at = utc_now()
try:
    if not scope_slug.strip() or not start_date or not end_date:
        raise ValueError("scope_slug, start_date, and end_date are required")
    bronze_root = Path(lakehouse_files_root) / bronze_folder
    candidates = []
    for manifest_path in sorted((bronze_root / "_manifests").rglob("*.json")):
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        scope = manifest.get("scope", {})
        report_day = manifest.get("report_day", "")
        if scope.get("kind") == scope_kind and scope.get("slug") == scope_slug and start_date <= report_day <= end_date:
            candidates.append(manifest)
    manifests = select_latest_manifests(candidates)

    combined = NormalizedBatch()
    source_files = 0
    for manifest in manifests:
        context = SourceContext(
            scope_kind=scope_kind,
            scope_slug=scope_slug,
            report_type=manifest["report_type"],
            report_day=manifest["report_day"],
            ingestion_id=manifest["ingestion_id"],
            ingested_at=manifest["ingested_at"],
        )
        if manifest["status"] == "no_data":
            combined.record_snapshot(context, status="no_data")
            continue
        for item in manifest.get("files", []):
            source_files += 1
            relative_path = item["path"]
            content = (bronze_root / relative_path).read_bytes()
            combined.extend(normalize_ndjson(content, SourceContext(
                scope_kind=context.scope_kind,
                scope_slug=context.scope_slug,
                report_type=context.report_type,
                report_day=context.report_day,
                ingestion_id=context.ingestion_id,
                ingested_at=context.ingested_at,
                source_path=relative_path,
            )))

    combined.finalize()
    frames = create_dataframes(spark, combined)
    merge_dataframes(spark, frames, schema=silver_schema, snapshots=combined.snapshots)
    row_counts = {name: frame.count() for name, frame in frames.items()}
    quarantine_count = row_counts.get("silver_quarantine", 0)
    details = {"entity_type": entity_type, "manifests": len(manifests), "source_files": source_files, "table_rows": row_counts, "quarantine_rows": quarantine_count}
    write_audit("succeeded", started_at, "passed" if quarantine_count == 0 else "warning", details)
except Exception as error:
    try:
        write_audit("failed", started_at, "failed", {"entity_type": entity_type, "error_type": type(error).__name__})
    finally:
        raise
